# 1. Import Packages

In [ ]:
# pip install kneed

In [ ]:
# pip install rasterio

In [ ]:
import numpy as np
import os
import pandas as pd
import geopandas as gpd
import scipy
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_samples, silhouette_score, calinski_harabasz_score
import timeit
import math
import sklearn
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer
from sklearn.cluster import KMeans
import kneed
from osgeo import gdal
from google.colab import files
import geemap
import ee
from google.colab import drive
import rasterio as rio
from rasterio import merge
# Installs geemap package
import subprocess
try:
    import geemap
except ImportError:
    print('geemap package not installed. Installing ...')
    subprocess.check_call(["python", '-m', 'pip', 'install', 'geemap'])
import sys


# 2. Process Data Extracted from Google Earth Engine

## 2.Note
Download raster(s) of residual greenness extracted from Google Earth Engine. This should be in the user's Google Drive.

## 2.1 Extract time series from Google Earth Engine rasters

In [ ]:
## Change directory to location of study area data
os.getcwd()
os.chdir(path_to_file) #INPUT: path to the file that contains the residual greenness rasters from GEE

## set up time range of study
start_year = YR1 #INPUT: first year of the time series
end_year = YR2 #INPUT: last year of the time series

#identify files to merge
# the study area was too large for GEE to download as one raster, so this code provides a function to merge the separate rasters provided
file1 = 'StudyArea_residualGreenness_file1.tif' #INPUT: file that contains the residual greenness raster 1 from GEE
file2 = 'StudyArea_residualGreenness_file2.tif' #INPUT: file that contains the residual greenness raster 2 from GEE
raster_file_list = [file1, file2]
#file name of merged images
mergedName = file1[0:-6] + "_merged.tif" #this removes the "#.tif" from the input files and replaces it with "merged.tif"

##function to merge rasters together
def merge_rasters(raster_file_list): ## Adding width and height as custom parameters if want to change the size of raster
    output_file = os.path.join(mergedName)
    ds_lst = list()
    for raster in raster_file_list:
        ds = gdal.Warp('', raster, format='vrt', dstNodata=0,
                       dstSRS="+proj=longlat +datum=WGS84 +no_defs +ellps=WGS84 +towgs84=0,0,0")
        ds_lst.append(ds)
        del ds
    dataset = gdal.BuildVRT('', ds_lst, VRTNodata=0, srcNodata=0)
    ds1 = gdal.Translate(output_file, dataset)
    del ds1
    del dataset
    return output_file

#enact function and merge rasters
output_file = mergedName
vrt_file = "merged.vrt"
gdal.BuildVRT(vrt_file,raster_file_list)

# Translate VRT to TIFF
gdal.Translate(output_file,vrt_file)

##get longitude and latitude data to input into CSV
file_name = output_file
with rio.open(file_name) as src:
    band1 = src.read(1)
    print('Band1 has shape', band1.shape)
    height = band1.shape[0]
    width = band1.shape[1]
    cols, rows = np.meshgrid(np.arange(width), np.arange(height))
    xs, ys = rio.transform.xy(src.transform, rows, cols)
    lons= np.array(xs)
    lats = np.array(ys)
    print('lons shape', lons.shape)

##flatten longitude and latitude data into 1D array
lons_f = lons.flatten()
lats_f = lats.flatten()

## get array of merged raster
ras_comb = rio.open(output_file) #INPUT: file name of merged raster file
ras_comb_arr = ras_comb.read()

## reshape raster array to have correct dimensions for a dataframe
years = ras_comb_arr.shape[0]
pixels = ras_comb_arr.shape[1] * ras_comb_arr.shape[2]
ras_comb_arr_reshape = ras_comb_arr.reshape((years,pixels))

#create dataframe from reshaped raster array and transpose to be in correct format for clustering
rasDF = pd.DataFrame(ras_comb_arr_reshape).transpose()

#remove NAN values from dataframe
rasDF_clean = rasDF.dropna()

##add longitude and latitude to dataframe
rasDF["lon"] = lons_f
rasDF["lat"] = lats_f

##save dataframe as CSV
output_csv = output_file[0:-4] + ".csv"
rasDF.to_csv(output_csv)

## 2.2 Clean and scale time series data

In [ ]:
## Set working directory to location of file
os.getcwd()
os.chdir(path_to_file) #INPUT: path to the CSV file that contains the study area residual greenness time series

## import csv file with data and coordinates
geoDF = pd.read_csv(output_csv)

##remove any NaN values from dataframe (although there should not be any) and drop lon, lat, and copied index columns for time series dataframe
geoDF_clean = geoDF.dropna()
geoDF_ts = geoDF_clean.drop(columns = ["lon","lat","Unnamed: 0","index"])

## scale data (by time series) using min-max normalization
scaled_arr = MinMaxScaler().fit_transform(geoDF_ts.transpose()) #transpose dataframe so each time series is scaled by its own min and max
##convert back to dataframe and original format
df_T_scaled = pd.DataFrame(scaled_arr, index = geoDF_ts.transpose().index)
# convert data back to rows
df_scaled = df_T_scaled.transpose()
ndvi_arr = df_scaled.to_numpy()

# 3. Time Series Clustering

## 3.1 Determine appropriate parameters for the algorithm

In [ ]:
## Use kneed algorithm to implement kneedle function to find point of maximum curvature
kn = kneed.KneeLocator(
    x,
    y,
    #S = 10,
    curve='convex',
    direction='decreasing',
    interp_method='interp1d',
    #interp_method='polynomial'
)

print("Number of clusters from kneedle algorithm: ",kn.knee)

In [ ]:
## Finding the Optimal Number of Clusters through within-cluster distance to center to check that the kneedle algorithm looks like it found the elbow
ssd = []
range_n_clusters = np.arange(2,100)
for num_clusters in range_n_clusters:
    print(num_clusters)
    kmeans = KMeans(n_clusters = num_clusters, random_state = 0)
    kmeans.fit(ndvi_arr)

    ssd.append(kmeans.inertia_)
plt.plot(ssd)
plt.xlabel("Clusters")
plt.ylabel("SSD")
#plt.savefig("clusterSSD_2_100.png") #INPUT: file name to save elbow plot
ssd_df = pd.DataFrame(ssd)
#ssd_df.to_csv("clusterSSD_2_100.csv") #INPUT: file name to save elbow plot CSV

In [4]:
## define k clusters
k_clusters = kn.knee ## check with the elbow plot that this number is reasonable
## set up kmeans parameters; these are adjustable
max_iter = 500 ## maximum number of iterations of the k-means algorithm for a single run.
n_init = 1 ## number of times the k-means algorithm is run with different centroid seeds.
tol = 1e-6 ## tolerance with regards to difference in the cluster centers of two consecutive iterations to declare convergence.
random_state = 0 ## random state so that the algorithm reproduces the same result each time it is run

## figure out appropriate number of iterations using k clusters; we are starting with one initialization to reduce computation time
km_iters = KMeans(n_clusters=k_clusters, max_iter=max_iter, n_init = n_init, tol = tol,
                          random_state=random_state)
km_iters_fit = km_iters.fit(ndvi_arr)
# ## see how many iterations
print(km_iters_fit.n_iter_)
iters = km_iters_fit.n_iter_

## 3.2 Run the clustering

In [3]:
## run k-means clustering with the number of iterations determined by the above step
n_init = 3 ## adjust n_init to 3 to reduce possibility of local minima
km = KMeans(n_clusters=k_clusters, max_iter=max_iter, n_init = n_init,
                          random_state=0)
km_fit = km.fit(ndvi_arr)

In [ ]:
## Look at properties of the clustering
km_fit.cluster_centers_.shape
np.unique(km_fit.labels_)

## 3.3 Evaluate Model Performance

In [ ]:
## Calculate the silhouette score to determine the "effectiveness" of the clustering
## -1 = data points are closer to the centroids of other clusters than their assigned cluster
## 0 = little cluster separability
## 1 = data points are close to the centroids of their own cluster and far from centroids of other clusters
labels = km_fit.labels_
sil = silhouette_score(ndvi_arr, labels, random_state = 17)
print("Silhouette score: ", sil)
## NOTE: if data is too large, silhouette score may be computationally intensive
## can calculate silhouette score with subsample of the data
sample_size = pixels/10 ##this is adjustable
sil_sample = silhouette_score(ndvi_arr, labels, sample_size = sample_size, random_state = random_state)

## 3.4 Plot Clustering Results

In [ ]:
## Recreate dataframe and transpose it
df = pd.DataFrame(ndvi_arr.reshape(geoDF_ts.shape), columns=np.arange(start_year,end_year+1), index=geoDF_ts.index) ##end year + 1 because the arange is not inclusive of the last number
df_T = df.transpose()


In [1]:
## code taken from from https://andrewm4894.com/2020/09/03/time-series-clustering-with-tslearn/
## Build helper df to map metrics to their cluster label
df_cluster = pd.DataFrame(list(zip(df_T.columns, labels)), columns=['pixel', 'cluster'])

## Make some helper dictionaries and lists
cluster_metrics_dict = df_cluster.groupby(['cluster'])['pixel'].apply(lambda x: [x for x in x]).to_dict()
cluster_len_dict = df_cluster['cluster'].value_counts().to_dict()
clusters_dropped = [cluster for cluster in cluster_len_dict if cluster_len_dict[cluster]==1]
clusters_final = [cluster for cluster in cluster_len_dict if cluster_len_dict[cluster]>1]
clusters_final.sort()


### 3.4a Create plots of the scaled data

In [2]:
## Create plots of all the mean cluster trajectories (Note that these plots are based on min-max scaled values)
num_clusters = k_clusters
#colors = ["#eae200", "#7fccf6","#2bd78d","#510068","#f4a03c","#ff8bf2","#3da3a5","#d1b9e7","#b744eb","#8b86fc","#b1f892","#859c00","#5f5f5f","#007092",
#          "#59c626","#f24a71","#bfc71b","#b8b8b8","#91063c","#e7788c"] #INPUT: set a color palette with colors for each cluster if desired; example given here for 20 clusters
## this sets up a set of subplots that should contain enough boxes for all of the clusters
rows = int(math.sqrt(num_clusters))
cols = math.ceil(math.sqrt(num_clusters))

plt.Figure(figsize =(10,5))
fig, axs = plt.subplots(int(math.sqrt(num_clusters)), math.ceil(math.sqrt(num_clusters)), figsize = (10,10))
fig.tight_layout()
cluster_number = 0
for i in np.arange(0,rows):
    for j in np.arange(0,cols):
        mean_ts = df_T[cluster_metrics_dict[cluster_number]].mean(axis = 1)
        print(cluster_number)
        #axs[i,j].plot(mean_ts, colors[cluster_number]) ## if using custom color palette
        axs[i,j].plot(mean_ts) ## if not using custom color palette
        axs[i,j].set_title("Cluster "+ str(cluster_number+1))
        axs[i,j].set_ylim(0,1)
        cluster_number = cluster_number + 1
        if cluster_number == num_clusters:
            break
#plt.savefig("StudyArea_kmeans_clusterMeans.png") #INPUT: file name for figure of mean cluster time series

In [5]:
## Create plots of all the mean cluster trajectories with one standard deviation (Note that these plots are based on min-max scaled values)
num_clusters = k_clusters
#colors = ["#eae200", "#7fccf6","#2bd78d","#510068","#f4a03c","#ff8bf2","#3da3a5","#d1b9e7","#b744eb","#8b86fc","#b1f892","#859c00","#5f5f5f","#007092",
#          "#59c626","#f24a71","#bfc71b","#b8b8b8","#91063c","#e7788c"] #INPUT: set a color palette with colors for each cluster if desired; example given here for 20 clusters
## this sets up a set of subplots that should contain enough boxes for all of the clusters
rows = int(math.sqrt(num_clusters))
cols = math.ceil(math.sqrt(num_clusters))
plt.Figure(figsize =(10,8))
fig, axs = plt.subplots(int(math.sqrt(num_clusters)), math.ceil(math.sqrt(num_clusters)), figsize = (10,10))
fig.tight_layout()
cluster_number = 0
for i in np.arange(0,rows):
    for j in np.arange(0,cols):
        df_conf = df_T[cluster_metrics_dict[cluster_number]].transpose()
        mean, lower, upper = [],[],[]
        for col in df_conf.columns:
            a = df_conf[col]
            m = np.mean(a)
            ml = m-np.std(a)
            mu = m + np.std(a)
            mean.append(m)
            lower.append(ml)
            upper.append(mu)
        mean_ts = df_T[cluster_metrics_dict[cluster_number]].mean(axis = 1)
        print(cluster_number)
        # axs[i,j].plot(mean,color=colors[cluster_number], label='mean') ## if using custom color palette
        axs[i,j].plot(mean, label='mean') ## if not using custom color palette
        #axs[i,j].fill_between(list(range(len(mean))), upper, lower, color=colors[cluster_number], alpha=0.3) ## if using custom color palette
        axs[i,j].fill_between(list(range(len(mean))), upper, lower, alpha=0.3) ## if not using custom color palette
        axs[i,j].set_title("Cluster "+ str(cluster_number+1))
        axs[i,j].set_ylim(-0.1,1.1)
        cluster_number = cluster_number + 1
        if cluster_number == num_clusters:
            break
#plt.savefig("StudyArea_kmeans_clusterStdev.png") #INPUT: file name for plot of cluster mean time series and their standard deviations

### 3.4b Create plots of the original (unscaled) data

In [ ]:
## reset index to 0 of dataframe
geoDF_reset = geoDF_clean.reset_index(drop=True)
#add cluster information to dataframe
geoDF_reset['cluster'] = df_cluster["cluster"]

## This may differ depending on the data format; essentially we want columns representing pixel number, each year, longitude, and latitude
##drop "unnamed:0" column
geoDF_reset = geoDF_reset.drop(columns = "Unnamed: 0")
##rename "index" column to "pixel"
geoDF_reset = geoDF_reset.rename(columns = {"index":"pixel"})

## convert column names to years
year = start_year
for column in geoDF_reset.columns[1:-2]: #INPUT: this should be the columns represented by the years
    geoDF_reset = geoDF_reset.rename(columns = {str(column): str(year)})
    year = year + 1

In [ ]:
## grab file with clusters and extract time series data
clusterDF = geoDF_reset
cDF_T = clusterDF.drop(columns = ["pixel","lat","lon","cluster"]).transpose()
labels = clusterDF["cluster"]

In [ ]:
## code taken from from https://andrewm4894.com/2020/09/03/time-series-clustering-with-tslearn/
## Build helper df to map metrics to their cluster label
df_cluster = pd.DataFrame(list(zip(cDF_T.columns, labels)), columns=['pixel', 'cluster'])

## Make some helper dictionaries and lists
cluster_metrics_dict_unsc = df_cluster_unsc.groupby(['cluster'])['pixel'].apply(lambda x: [x for x in x]).to_dict()
cluster_len_dict_unsc = df_cluster_unsc['cluster'].value_counts().to_dict()
clusters_dropped_unsc = [cluster for cluster in cluster_len_dict_unsc if cluster_len_dict_unsc[cluster]==1]
clusters_final_unsc = [cluster for cluster in cluster_len_dict_unsc if cluster_len_dict_unsc[cluster]>1]
clusters_final_unsc.sort()

In [6]:
## Create plots of all the mean cluster trajectories with one standard deviation
num_clusters = k_clusters
#colors = ["#eae200", "#7fccf6","#2bd78d","#510068","#f4a03c","#ff8bf2","#3da3a5","#d1b9e7","#b744eb","#8b86fc","#b1f892","#859c00","#5f5f5f","#007092",
#          "#59c626","#f24a71","#bfc71b","#b8b8b8","#91063c","#e7788c"] #INPUT: set a color palette with colors for each cluster if desired; example given here for 20 clusters
## this sets up a set of subplots that should contain enough boxes for all of the clusters
rows = int(math.sqrt(num_clusters))
cols = math.ceil(math.sqrt(num_clusters))
plt.Figure(figsize =(10,8))
fig, axs = plt.subplots(int(math.sqrt(num_clusters)), math.ceil(math.sqrt(num_clusters)), figsize = (10,10))
fig.tight_layout()
cluster_number = 0
for i in np.arange(0,rows):
    for j in np.arange(0,cols):
        df_conf = cDF_T[cluster_metrics_dict_unsc[cluster_number]].transpose()
        mean, lower, upper = [],[],[]
        for col in df_conf.columns:
            a = df_conf[col]
            m = np.mean(a)
            ml = m-np.std(a)
            mu = m + np.std(a)
            mean.append(m)
            lower.append(ml)
            upper.append(mu)
        mean_ts = cDF_T[cluster_metrics_dict_unsc[cluster_number]].mean(axis = 1)
        print(cluster_number)
        axs[i,j].plot(mean,color=colors[cluster_number], label='mean')## if using custom color palette
        axs[i,j].plot(mean, label='mean')## if not using custom color palette
        axs[i,j].fill_between(list(range(len(mean))), upper, lower, color=colors[cluster_number], alpha=0.3) ## if using custom color palette
        axs[i,j].fill_between(list(range(len(mean))), upper, lower, alpha=0.3) ## if not using custom color palette
        axs[i,j].set_title("Cluster "+ str(cluster_number+1))
        axs[i,j].set_ylim(-0.2,0.35)
        cluster_number = cluster_number + 1
        if cluster_number == num_clusters:
            break
fig.text(0.5,-0.01,'Year', ha='center', fontsize = 12)
fig.text(-0.03, 0.5, 'Residual Greenness', va='center', rotation='vertical', fontsize = 12)
#plt.savefig("StudyArea_kmeans_clusterStdev_unscaled.png",bbox_inches='tight') #INPUT: file name for unscaled plot of mean cluster time series and standard deviations

## 3.5 Map the pixels back into space with the cluster assignment

In [ ]:
##save to CSV
geoDF_reset.to_csv('StudyArea_clustered_kmeans.csv', index = False) #INPUT: file name to save the time series plus their cluster assignment

In [ ]:
#import and read base raster file
os.getcwd()
os.chdir(path_to_file) ## path to raster file of the merged residual greenness raster files
ras_file = output_file #INPUT: file name of the merged raster file
ras = rio.open(ras_file)

## convert csv to geodataframe
gdf = gpd.GeoDataFrame(geoDF_reset, geometry=gpd.points_from_xy(geoDF_reset['lon'], geoDF_reset['lat']), crs="EPSG:4326")

## create a rasterized version of the shapefile
## create tuples of geometry, value pairs, where value is the attribute value you want to burn
shp = gdf
geom_value = ((geom,value) for geom, value in zip(shp.geometry, shp['cluster']))

## Rasterize vector using the shape and transform of the raster
rasterized = features.rasterize(geom_value,
                                out_shape = ras.shape,
                                transform = ras.transform,
                                all_touched = True,
                                fill = -5,   # background value
                                merge_alg = MergeAlg.replace,
                                dtype = int16)
# Save rasterized vector as a raster
outputFile = "StudyArea_OR_clustered_kmeans.tif" #INPUT: file name for raster of clusters
with rio.open(
        outputFile, "w", ##study area
        driver = "GTiff",
        crs = ras.crs,
        transform = ras.transform,
        dtype = rio.uint8,
        count = 1,
        width = ras.width,
        height = ras.height) as dst:
    dst.write(rasterized, indexes = 1)

# 4. LandTrendr Time Series Segmentation

## 4.1 Get mean time series of each cluster

In [7]:
## get pixel-wise time series
clusterDF_ts = clusterDF[clusterDF.columns[1:-2]] #INPUT: this should be the columns represented by the years
clusterDF_ts["cluster"] = clusterDF["cluster"]

In [8]:
## get the mean time series of each cluster
clusterMeans = clusterDF_ts.groupby(['cluster']).mean()

In [ ]:
## save the mean cluster time series as a CSV
clusterMeans.to_csv("StudyArea_kmeans_clusterMeans.csv") #INPUT: file name for CSV of mean cluster time series

In [ ]:
## save the mean cluster time series as a shapefile
clusterMeans_coords = clusterMeans
clusterMeans_coords["lat"] = lats_f[0:k_clusters]
clusterMeans_coords["lon"] = lons_f[0:k_clusters]
clusterMeans_coords_gdf = gpd.GeoDataFrame(clusterMeans_coords, geometry=gpd.points_from_xy(clusterMeans_coords['lon'], clusterMeans_coords['lat']), crs="EPSG:4326")
clusterMeans_coords_gdf.to_file("StudyArea_kmeans_clusterMeans_landtrendrShapefile.shp") #INPUT: file name to save shapefile with mean cluster time series

## save just the coordinates as a shapefile
coords_gdf = clusterMeans_coords_gdf[clusterMeans_coords_gdf.columns[-3:]] ##extract lat, long, and geometry from the geodataframe
coords_gdf.to_file("StudyArea_kmeans_clusterMeans_landtrendrCoords.shp") #INPUT: file name to save adjusted shapefile with just coordinates for mean cluster time series

### 4.Note.1
**(a)** Upload the shapefile with the mean cluster time series ("StudyArea_kmeans_clusterMeans_landtrendrShapefile.shp") to Google Drive \\
**(b)** Upload the shapefile with just the coordinates of the mean cluster time series grid cells ("StudyArea_kmeans_clusterMeans_landtrendrCoords.shp") as an asset in Google Earth Engine ([Manage assets](https://developers.google.com/earth-engine/guides/manage_assets)).

## 4.2 Conduct LandTrendr through Google Earth Engine API

In [ ]:
##connect to GEE account
ee.Authenticate()
#https://developers.google.com/earth-engine/guides/python_install
ee.Initialize(project="ee-project") #INPUT: name of Google Earth Engine cloud project

## set recursion limit
print(sys.getrecursionlimit())
sys.setrecursionlimit(1000000000)

1000


In [ ]:
## set up aoi
aoi_fc = ee.FeatureCollection('projects/ee-project/assets/StudyArea_kmeans_clusterMeans_landtrendrCoords') #INPUT: name of Google Earth Engine cloud project and path to file
aoi_coord1 = aoi_fc.getInfo()["features"][0]["geometry"]["coordinates"]
aoi_coord2 = aoi_fc.getInfo()["features"][-1]["geometry"]["coordinates"]

aoi_line = ee.Geometry.LineString([aoi_coord1,aoi_coord2])

## import dataframe of cluster means and grid coords
data = gpd.read_file("/content/drive/MyDrive/StudyArea_kmeans_clusterMeans_landtrendrShapefile.shp") #INPUT: file name for shapefile of mean cluster time series
data.head()

In [ ]:
## change magnitude of data so that decimals don't get lost
data[data.columns[0:-3]] = data[data.columns[0:-3]]*1e6
data = data.drop(columns = ["lat","lon"])
data.head()

## convert data to feature collection
fc = geemap.geopandas_to_ee(data)

## create image collection of the cluster means
images = []
for year in np.arange(start_year,end_year+1):
  yr = str(year)
  date = yr+"-01-01"
  image = fc.reduceToImage([yr], ee.Reducer.first()).set("system:time_start",ee.Date(date).millis())
  images.append(image)
imageColl = ee.ImageCollection(images)


In [ ]:
## set up LandTrendr parameters
maxSegments=6
spikeThreshold=0.9
vertexCountOvershoot=3
preventOneYearRecovery = True
recoveryThreshold=0.25
pvalThreshold=0.05
bestModelProportion=0.75
minObservationsNeeded=10

## setup LandTrendr
startYear_Num = start_year
endYear_Num   = end_year;

##run LandTrendr
lt = ee.Algorithms.TemporalSegmentation.LandTrendr(
  timeSeries=imageColl,
  maxSegments=maxSegments,
  spikeThreshold=spikeThreshold,
  vertexCountOvershoot=vertexCountOvershoot,
  preventOneYearRecovery=preventOneYearRecovery,
  recoveryThreshold=recoveryThreshold,
  pvalThreshold=pvalThreshold,
  bestModelProportion=bestModelProportion,
  minObservationsNeeded=minObservationsNeeded)

##extract results from landtrendr
ltlt = lt.select(['LandTrendr'])
years = ltlt.arraySlice(0, 0, 1)
observed = ltlt.arraySlice(0, 1, 2)
rmse = lt.select('rmse')
fitted = ltlt.arraySlice(0, 2, 3)
vertexMask = ltlt.arraySlice(0, 3, 4)
vertices = ltlt.arrayMask(vertexMask)

leftList  = vertices.arraySlice(1, 0, -1)  #slice out the vertices as the start of segments
rightList = vertices.arraySlice(1, 1)   # slice out the vertices as the end of segments
startYear = leftList.arraySlice(0, 0, 1)   #get year dimension of LT data from the segment start vertices
startVal  = leftList.arraySlice(0, 2, 3)      # get FITTED index data from LT at the segment start vertices
endYear   = rightList.arraySlice(0, 0, 1)     # get year dimension of LT data from the segment end vertices
endVal    = rightList.arraySlice(0, 2, 3)     # get FITTED index data from LT at the segment end vertices
dur       = endYear.subtract(startYear)      # subtract the segment start year from the segment end year to calculate the duration of segments
mag       = endVal.subtract(startVal)        # substract the segment start index value from the segment end index value to calculate the delta of segments
rate      = mag.divide(dur)

In [ ]:
## export LandTrendr Results
years = ee.List.sequence(startYear_Num, endYear_Num)
def getYearStr(year):
  return(ee.String('yr_').cat(ee.Algorithms.String(year).slice(0,4)))

yearsStr = years.map(getYearStr)
fittedStack = fitted.arrayFlatten([['fittedMeans'], yearsStr]).toShort();
print(fittedStack.bandNames().getInfo())
fittedStack_ex = fittedStack
## this will export the LandTrendr-processed mean cluster time series to Google Drive
drive = ee.batch.Export.image.toDrive(
            image = fittedStack,
            description = "LandTrendr_clusterMeans", #INPUT: file name for LandTrendr time series
            crs = "EPSG:4326",
            region = aoi_line,
            scale = 30
        )
drive.start()

['fittedMeans_yr_1985', 'fittedMeans_yr_1986', 'fittedMeans_yr_1987', 'fittedMeans_yr_1988', 'fittedMeans_yr_1989', 'fittedMeans_yr_1990', 'fittedMeans_yr_1991', 'fittedMeans_yr_1992', 'fittedMeans_yr_1993', 'fittedMeans_yr_1994', 'fittedMeans_yr_1995', 'fittedMeans_yr_1996', 'fittedMeans_yr_1997', 'fittedMeans_yr_1998', 'fittedMeans_yr_1999', 'fittedMeans_yr_2000', 'fittedMeans_yr_2001', 'fittedMeans_yr_2002', 'fittedMeans_yr_2003', 'fittedMeans_yr_2004', 'fittedMeans_yr_2005', 'fittedMeans_yr_2006', 'fittedMeans_yr_2007', 'fittedMeans_yr_2008', 'fittedMeans_yr_2009', 'fittedMeans_yr_2010', 'fittedMeans_yr_2011', 'fittedMeans_yr_2012', 'fittedMeans_yr_2013', 'fittedMeans_yr_2014', 'fittedMeans_yr_2015', 'fittedMeans_yr_2016', 'fittedMeans_yr_2017', 'fittedMeans_yr_2018', 'fittedMeans_yr_2019', 'fittedMeans_yr_2020', 'fittedMeans_yr_2021', 'fittedMeans_yr_2022', 'fittedMeans_yr_2023']


### 4.Note.2
Download file with the LandTrendr temporally segmented mean cluster time series ("LandTrendr_clusterMeans.tif") from Google Drive

## 4.3 Visualize LandTrendr Results

In [ ]:
##import image with fitted LandTrendr cluster time series
os.chdir(path_to_file) #INPUT: path to file containing LandTrendr-processed mean cluster time series
ras = "LandTrendr_clusterMeans.tif" #INPUT: file name for LandTrendr time series

## open image
LTfit = rio.open(ras)
bands = LTfit.read()

cmeans_LT = []
for band in np.arange(1, bands.shape[0]+1):
    vals = LTfit.read(int(band))
    cmeans_LT.append(vals/1e6) ## the values were scaled by 1e6 -> unscale here

## create dataframe
cmeans_LT_df = pd.DataFrame(cmeans_LT, index = np.arange(start_year,end_year+1))
cmeans_LT_df_T = cmeans_LT_df.transpose()



In [ ]:
## Create plots of all the mean cluster trajectories overlaid by LT (unscaled)
num_clusters = k_clusters
#colors = ["#eae200", "#7fccf6","#2bd78d","#510068","#f4a03c","#ff8bf2","#3da3a5","#d1b9e7","#b744eb","#8b86fc","#b1f892","#859c00","#5f5f5f","#007092",
#          "#59c626","#f24a71","#bfc71b","#b8b8b8","#91063c","#e7788c"] #INPUT: set a color palette with colors for each cluster if desired; example given here for 20 clusters
## this sets up a set of subplots that should contain enough boxes for all of the clusters
rows = int(math.sqrt(num_clusters))
cols = math.ceil(math.sqrt(num_clusters))
plt.Figure(figsize =(10,5))
fig, axs = plt.subplots(int(math.sqrt(num_clusters)), math.ceil(math.sqrt(num_clusters)), figsize = (10,10))
fig.tight_layout()
cluster_number = 0
for i in np.arange(0,rows):
    for j in np.arange(0,cols):
        mean_ts = cDF_T[cluster_metrics_dict[cluster_number]].mean(axis = 1)
        lt_ts = cmeans_LT_df[cluster_number]
        print(cluster_number)
        #axs[i,j].plot(mean_ts, colors[cluster_number]) #if using custom color palette
        axs[i,j].plot(mean_ts) #if not using custom color palette
        axs[i,j].plot(lt_ts, "k",linestyle='--')
        axs[i,j].set_title("Cluster "+ str(cluster_number+1))
        axs[i,j].set_ylim(-0.1,0.3)
        cluster_number = cluster_number + 1
        if cluster_number == num_clusters:
            break
#plt.savefig("StudyArea_OR_kmeans_clusterMeans_LandTrendr.png") #INPUT: file name for plot of landtrendr time series overlaid on mean cluster time series